# ICS 604: APPLIED DATA SCIENCE

## Mixture Models

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy as sp
from scipy import stats

matplotlib.rcParams['figure.figsize'] = [7, 3.5]

## Mixture Model Intuition

A helpful way to build intuition for a mixture model is to imagine you’re looking at data that clearly comes from different underlying groups, but you don’t know which data point belongs to which group. In the spending example, purchases made by males and females may follow different patterns, yet the dataset itself does not label them. If you apply a clustering method like k-means, you can still partition the data into groups based on similarity. However, this approach treats clusters as rigid assignments — each point belongs entirely to one cluster — without expressing uncertainty.

Mixture models take this a step further by assuming that the data is generated from a combination of probabilistic sources. Instead of just grouping points, you model each group as a probability distribution, often Gaussian. Each distribution has its own parameters, such as a mean and standard deviation, which describe the typical behavior of that group. A single observed data point is then viewed as having been generated by one of these distributions, but you don’t directly observe which one. Importantly, the model allows you to compute how likely it is that a point came from each distribution, rather than forcing a hard assignment.

To make this more concrete, imagine briefly that you could identify whether each purchase came from a male or female customer. In that case, estimating the parameters of each Gaussian would be straightforward: you would simply compute the mean and variance for each labeled group. The challenge — and the elegance — of mixture models lies in removing this assumption. Since the labels are hidden, the model must simultaneously infer both the parameters of the distributions and the probability that each data point belongs to each group. This interplay between uncertainty in assignments and estimation of parameters is what gives mixture models their flexibility and power.

In [ ]:
np.random.seed(45)

mean_males = 204
std_males = 31

mean_females = 257
std_females = 28

sales_males = np.random.normal(mean_males, std_males, 387)
sales_females = np.random.normal(mean_females, std_females, 422)

sales_total = np.concatenate([sales_males, sales_females])

plt.hist(sales_total, edgecolor='k', linewidth=1);

In [ ]:
plt.hist(sales_males, alpha=0.5, edgecolor='k', linewidth=0.5)
plt.hist(sales_females, alpha=0.5, edgecolor='k', linewidth=0.5);

In [ ]:
kde_total = sp.stats.gaussian_kde(sales_total, bw_method=0.2)

x_axis = np.arange(50, 450)
y_axis_total = kde_total.evaluate(x_axis)
plt.plot(x_axis, y_axis_total);

### Finite Mixture Model

When we visualize the overall data without labels, we often notice that the distribution has multiple peaks. In this case, the total sales distribution shows two distinct peaks, suggesting that it is actually a combination of two underlying distributions. This is what we mean by a “mixture”: the observed data is formed by blending multiple simpler distributions together. These peaks may not always be sharply visible in a histogram due to binning effects, but they often appear more clearly in a kernel density estimate (KDE), which provides a smoother view of the data.

The separation between these underlying distributions plays a key role in how easily we can identify them. If the means of the two distributions are far apart, the peaks become more distinct and the mixture structure is obvious. On the other hand, if the means are close together, the distributions overlap significantly, making it harder to distinguish between them. Finite mixture models are designed to handle this situation: even when the components are not clearly separated, they provide a framework for modeling the data as a combination of multiple probabilistic sources and estimating the contribution of each.

In [ ]:
mean_males2, std_males2 = 150, 31
mean_females2, std_females2 = 300, 28

sales_males2 = np.random.normal(mean_males2, std_males2, 387)
sales_females2 = np.random.normal(mean_females2, std_females2, 422)

sales_total2 = np.concatenate([sales_males2, sales_females2])

kde_total2 = sp.stats.gaussian_kde(sales_total2, bw_method=0.17)
y_axis2 = kde_total2.evaluate(x_axis)
plt.plot(x_axis, y_axis2);


In this setting, total sales can be viewed as a finite mixture of two underlying distributions. Rather than assuming all data points come from a single homogeneous source, we model the observed values as being generated by one of several distinct random processes. A mixture distribution, therefore, is the probability distribution of a random variable that arises from combining multiple other distributions. Each component contributes to the overall shape, and the final distribution reflects this blending — often revealing multiple peaks or regions of higher density.

Mixtures are not just a theoretical construct; they appear naturally in many real-world scenarios. For example, when analyzing spending on a digital marketplace like the Apple Store, prices for professional-grade applications can differ significantly from those aimed at general consumers. Apps designed for specialized users — such as doctors, stock traders, or frequent travelers — may cluster at higher price points, while apps for students or casual users tend to be cheaper. Similarly, in healthcare, recovery times can vary widely depending on treatment type or pre-existing conditions, leading to distinct groups such as fast and slow recoveries. In both cases, what looks like a single dataset is actually a combination of multiple subpopulations with different characteristics.

These underlying subpopulations can be thought of as clusters, where each mixture component corresponds to a group with its own statistical properties. Finite mixture models assume a fixed number of such components, but more advanced probabilistic approaches relax this constraint. Some models allow for a potentially infinite number of components, adapting their complexity to the data. These flexible models are especially useful in domains like document organization or biological classification, where the true number of clusters — such as topics or species — is not known in advance.

In [ ]:
kde_males = sp.stats.gaussian_kde(sales_males, bw_method=0.5)
y_axis_males = kde_males.evaluate(x_axis)
plt.plot(x_axis, y_axis_males);

In [ ]:
kde_females = sp.stats.gaussian_kde(sales_females, bw_method=0.5)
y_axis_females = kde_females.evaluate(x_axis)
plt.plot(x_axis, y_axis_females, color='orange');

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(x_axis, y_axis_males)
plt.plot(x_axis, y_axis_females)

plt.subplot(1, 2, 2)
plt.plot(x_axis, y_axis_total);

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(x_axis, y_axis_males)
plt.plot(x_axis, y_axis_females)

plt.scatter(sales_males, np.zeros(len(sales_males)))
plt.scatter(sales_females, np.zeros(len(sales_females)) + 0.0005);

## Using k-Means to Identify the Two Clusters

If we momentarily step back into the simplified world where labels are recoverable, we can treat the total sales data as a combination of two groups: `sales_males` and `sales_females`. The dataset `sales_total` is simply their concatenation, containing observations from 387 males and 422 females. Even though the labels are not provided, we know there are exactly two underlying groups, which makes this a natural setting for clustering methods.

Using k-means, we can attempt to recover these two groups by assigning each data point to one of two clusters. The algorithm works by finding two centroids and grouping points based on proximity to those centroids. In this context, each entry in `sales_total` would be assigned a label — say 0 or 1 — which we can interpret as representing male or female. Of course, k-means does not “know” anything about gender; it simply partitions the data into two clusters that minimize within-cluster variance.

In [ ]:
y_true = np.concatenate([np.zeros(387), np.ones(422)])
y_true[380:394]

In [ ]:
from sklearn.cluster import KMeans

X = sales_total.reshape(-1, 1)

kmeans = KMeans(n_clusters=2, n_init=5).fit(X)
y_predicted = kmeans.predict(X)

kmeans.cluster_centers_

In [ ]:
y_predicted[0:30]

In [ ]:
y_predicted[-30:]

In [ ]:
y_true[-30:]

In [ ]:
y_predicted = 1 - y_predicted
print(sum(y_predicted == y_true) / len(y_predicted))

In [ ]:
pred_males = np.where(y_predicted == 0)
pred_females = np.where(y_predicted == 1)

pred_males_error = np.where((y_predicted == 0) & (y_true == 1))
pred_females_error = np.where((y_predicted == 1) & (y_true == 0))

In [ ]:
plt.figure(figsize=(16, 4))

# REAL DATA
plt.scatter(sales_total[:len(sales_males)], np.zeros(len(sales_males)), color='blue')
plt.scatter(sales_total[len(sales_males):], np.zeros(len(sales_females))+0.01, color='orange')

# PREDICTED DATA
plt.scatter(sales_total[pred_males], np.zeros(len(y_predicted[pred_males]))+0.05, color='blue')
plt.scatter(sales_total[pred_females], np.zeros(len(y_predicted[pred_females]))+0.06, color='orange')

# ERRORS IN PREDICTION
plt.scatter(sales_total[pred_males_error], np.zeros(len(y_predicted[pred_males_error]))+0.1, color='red')
plt.scatter(sales_total[pred_females_error], np.zeros(len(y_predicted[pred_females_error]))+0.11, color='green')

plt.yticks([], []);

### Impracticality of the k-Means

k-means is a useful starting point, but it comes with structural assumptions that can become limiting. It works best when the data forms well-separated “blobs,” where each cluster is roughly spherical and centered around a mean. The algorithm partitions space into regions based on distance to these centroids, which effectively creates circular (or, in higher dimensions, hyperspherical) boundaries. When the true underlying groups follow this kind of geometry, k-means can recover them quite efficiently.

However, real-world data rarely behaves so neatly. When clusters overlap — even if they are still roughly blob-shaped — k-means struggles because it enforces hard boundaries. Every point must belong entirely to one cluster or the other, regardless of how ambiguous its position might be. This becomes especially problematic when the two groups have significant overlap, as is often the case in mixture distributions. In such scenarios, k-means provides no measure of confidence or uncertainty; a point near the boundary is treated the same as one deep inside a cluster.

What we need instead is an approach that explicitly models this uncertainty. Rather than forcing a binary decision, we want a method that can say, for example, that a particular data point has a 70% chance of belonging to one group and a 30% chance of belonging to another. This is precisely the motivation behind probabilistic clustering methods like mixture models. By allowing soft assignments and incorporating the idea that data is generated from multiple overlapping distributions, they provide a much more flexible and realistic representation of complex datasets.

## Modeling a Mixture Distribution

Up to this point, we made a convenient assumption: we knew exactly how many data points came from each group (males and females). In reality, that information is almost never available. A more faithful model should treat group membership itself as random. Instead of fixing the counts, we introduce a parameter that governs the proportion of each group in the population. For example, we might assume that each individual has some probability $p$ of being male and $1−p$ of being female, and then sample group membership from a Bernoulli (or binomial, for many samples) distribution. This turns the group labels into latent variables — unobserved quantities that the model must infer.

With this in place, we can describe a proper generative process for the mixture distribution. First, for each data point, we randomly choose a component (e.g., male or female) according to the mixing proportion $p$. Then, conditional on that choice, we generate the observed value (such as spending) from the corresponding distribution — often modeled as a Gaussian with its own mean and variance. Repeating this process for all individuals produces a dataset that naturally blends the two groups together, without ever explicitly labeling them in the final observations.

This formulation highlights what makes mixture models powerful: both the component assignments and the parameters of the distributions are treated as unknowns. The model must simultaneously estimate the mixing proportion (how many points come from each group) and the characteristics of each component (their means and variances). By framing the problem this way, we move from a rigid, label-dependent view of the data to a probabilistic one where uncertainty is built into every step of the data-generating process.

### Defining the Mixture Distribution for Spending Example

<center><img src="https://www.dropbox.com/scl/fi/pqynawfrvawjykevkrv47/generative_model.png?rlkey=mvpwjcgwrdokjuno2e5pg96pz&st=c63geuvq&dl=1" alt="drawing" style="width:700px"/>

In [ ]:
np.random.seed(42)
male_female_prop = 0.45

mean_males = 204
std_males = 31

mean_females = 257
std_females = 28

sales_males =  []
sales_females = []
cluster = []

for i in range(1000):
    gender = np.random.choice([0, 1], p=[male_female_prop, 1 - male_female_prop])
    
    if gender == 0:
        cluster.append(0) 
        male_sale = np.random.normal(mean_males, std_males) 
        sales_males.append(male_sale)
    else:
        cluster.append(1)
        female_sale = np.random.normal(mean_females, std_females) 
        sales_females.append(female_sale)
        
sales_total = np.concatenate([sales_males, sales_females])  

In [ ]:
kde_total = sp.stats.gaussian_kde(sales_total, bw_method=0.2)

x_axis_total = np.arange(50, 450)
y_axis_total_kde = kde_total.evaluate(x_axis_total)

plt.plot(x_axis, y_axis_total_kde);

In [ ]:
kde_males = sp.stats.gaussian_kde(sales_males, bw_method=0.5)

x_axis_males = np.arange(50, 450)
y_axis_males_kde = kde_males.evaluate(x_axis_males)

plt.plot(x_axis_males, y_axis_males_kde);

In [ ]:
kde_females = sp.stats.gaussian_kde(sales_females, bw_method=0.5)

x_axis_females = np.arange(100, 400)
y_axis_females_kde = kde_females.evaluate(x_axis_females)

plt.plot(x_axis_females, y_axis_females_kde, color="orange");

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(x_axis_males, y_axis_males_kde)
plt.plot(x_axis_females, y_axis_females_kde)

plt.subplot(1, 2, 2)
plt.plot(x_axis, y_axis_total_kde);

### Decomposing the Mixture

To understand how a mixture distribution is constructed, it helps to think in terms of combining simpler building blocks. Suppose we have two underlying random variables: one for male spending and one for female spending, each modeled as a Gaussian distribution. Formally, we write

$$
\begin{aligned}
X_{male} &\sim \mathcal{N}(\mu_{male},\sigma_{male})\\
X_{female} &\sim \mathcal{N} (\mu_{female},\sigma_{female}).
\end{aligned}
$$
 
Each of these defines its own probability density function (pdf), which tells us how likely different values of $x$ are under that specific group.

In a mixture setting, any observed data point $x$ is evaluated under both distributions. That is, we can compute $\mbox{pdf}_{male}(x)$ and $\mbox{pdf}_{female}(x)$ using the Gaussian density formula. However, the key idea is that the overall (mixture) density is not just a simple sum of these two values — it is a *weighted* sum. Each component contributes according to how prevalent it is in the population. These weights are often called mixing coefficients and typically sum to 1.

Concretely, if $\pi$ represents the proportion of males in the data, then the mixture distribution is given by:

$$
\mbox{pdf}(x) = \pi\cdot\mbox{pdf}_{male}(x) + (1 - \pi)\cdot\mbox{pdf}_{female}(x).
$$

This weighting naturally reflects the relative sample sizes: if there are more males than females, the male distribution contributes more heavily to the combined density. In this sense, normalization by sample size is built directly into the mixing proportions.

This decomposition shows why mixture models are so expressive. Instead of forcing all data to conform to a single distribution, they allow multiple distributions to “share responsibility” for explaining each point. A value of $x$ near the overlap of the two Gaussians might have significant probability under both components, reinforcing the idea that mixture models capture uncertainty and ambiguity in a principled, probabilistic way.

In [ ]:
mean_males = 204
std_males = 31

x_axis = np.arange(50, 450)
y_axis_males_theoretical = sp.stats.norm.pdf(x_axis, mean_males, std_males)

plt.plot(x_axis, y_axis_males);

In [ ]:
mean_females = 257
std_females = 28

x_axis = np.arange(50, 450)
y_axis_females_theoretical = sp.stats.norm.pdf(x_axis, mean_females, std_females)

plt.plot(x_axis, y_axis_females_theoretical);

In [ ]:
plt.plot(x_axis, y_axis_males_theoretical)
plt.plot(x_axis, y_axis_females_theoretical);

In [ ]:
plt.plot(x_axis, y_axis_males_theoretical, label='male')
plt.plot(x_axis, y_axis_females_theoretical, label='female')
plt.plot(x_axis, y_axis_total_kde, label="kde-based")
plt.legend();

In [ ]:
len(sales_males), len(sales_females) 

In [ ]:
y_axis_total_theoretical = ((len(sales_males) / 1000) * y_axis_males_theoretical 
                                + (len(sales_females) / 1000) * y_axis_females_theoretical)

plt.plot(x_axis, y_axis_total_kde, label="kde-based")
plt.plot(x_axis, y_axis_total_theoretical, label= "Theoretical")

plt.legend();

### Question

Suppose we have access to the parameters of two populations, $P_1$ and $P_2$, specifically their means $(\mu_1, \mu_2)$ and standard deviations $(\sigma_1, \sigma_2)$. Given a new observation $x_1$, the goal is to determine which population it most likely originated from.

To do this, we consider how likely the value $x_1$ is under each population’s distribution. Since each population is modeled as a Gaussian, we can evaluate the probability density of $x_1$ under both $P_1$ and $P_2$ using their respective parameters. This gives us a way to quantify how well each distribution explains the observed point.

By comparing these probability density values, we can assess which population is a better explanation for $x_1$. The population that assigns a higher density to the point is considered more consistent with its generation. In this way, the decision is based on relative likelihoods under the two distributions rather than any fixed distance or rule.

In the visualization, the two theoretical density curves represent $P_1$ and $P_2$, and the point $x_1=300$ is plotted for reference. Its position relative to the two curves reflects how the likelihood comparison is made visually.

In [ ]:
plt.plot(x_axis, y_axis_males_theoretical, label="male")
plt.plot(x_axis, y_axis_females_theoretical, label="female")
plt.scatter(300, 0, color='red', marker="X", s=200)
plt.legend();

#### To be continued...